# Análise Exploratória do Brasileirão (Refatorada)

Este notebook contém uma versão refatorada da análise exploratória, focada em modularidade, legibilidade e redução de duplicidade de código.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações globais de plotagem
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Carregamento e Limpeza Inicial

In [ ]:
def load_and_preprocess(filepath):
    df = pd.read_csv(filepath)
    # Exibe valores nulos iniciais para diagnóstico
    null_counts = df.isnull().sum().sort_values(ascending=False)
    print("Principais colunas com valores nulos:\n", null_counts.head(10))
    return df

df_raw = load_and_preprocess('mundo_transfermarkt_competicoes_brasileirao_serie_a.csv')

## 2. Funções Utilitárias de Processamento

In [ ]:
def calculate_points(df):
    """Calcula pontos ganhos por mandante e visitante."""
    df = df.copy()
    
    for role in ['mandante', 'visitante']:
        other = 'visitante' if role == 'mandante' else 'mandante'
        conds = [
            df[f'gols_{role}'] > df[f'gols_{other}'],
            df[f'gols_{role}'] == df[f'gols_{other}'],
            df[f'gols_{role}'] < df[f'gols_{other}']
        ]
        df[f'pontos_{role}'] = np.select(conds, [3, 1, 0])
    
    return df

def unify_team_data(df, extra_cols_mapping):
    """
    Unifica dados de mandante e visitante em uma visão por time.
    
    Args:
        df: DataFrame original.
        extra_cols_mapping: Dicionário {nome_unificado: (col_mandante, col_visitante)}
    """
    base_mapping = {
        'ano_campeonato': ('ano_campeonato', 'ano_campeonato'),
        'data': ('data', 'data'),
        'rodada': ('rodada', 'rodada'),
        'time': ('time_mandante', 'time_visitante'),
        'pontos_ganhos': ('pontos_mandante', 'pontos_visitante'),
        'gols_pro': ('gols_mandante', 'gols_visitante')
    }
    base_mapping.update(extra_cols_mapping)
    
    df_home = df[[v[0] for v in base_mapping.values()]].copy()
    df_home.columns = list(base_mapping.keys())
    
    df_away = df[[v[1] for v in base_mapping.values()]].copy()
    df_away.columns = list(base_mapping.keys())
    
    unified = pd.concat([df_home, df_away], ignore_index=True)
    return unified.sort_values(by=['ano_campeonato', 'time', 'data'])

## 3. Preparação dos Datasets de Análise

In [ ]:
# Dataset 1: Gestão (Técnicos e Idade) - 2007 a 2023
df_gestao = df_raw[(df_raw['ano_campeonato'] >= 2007) & (df_raw['ano_campeonato'] <= 2023)].copy()
df_gestao = df_gestao.dropna(subset=['tecnico_mandante', 'idade_media_titular_mandante'])
df_gestao = calculate_points(df_gestao)

# Dataset 2: Scout (Chutes e Faltas) - 2020 a 2023
df_scout = df_raw[(df_raw['ano_campeonato'] >= 2020) & (df_raw['ano_campeonato'] <= 2023)].copy()
df_scout = df_scout.dropna(subset=['chutes_mandante', 'faltas_mandante'])
df_scout = calculate_points(df_scout)

print(f"Dataset Gestão: {len(df_gestao)} partidas")
print(f"Dataset Scout: {len(df_scout)} partidas")

## 4. Análise de Desempenho (Perguntas de Pesquisa)

### A. Chutes ao gol vs. Rendimento

In [ ]:
mapping_scout = {
    'chutes_totais': ('chutes_mandante', 'chutes_visitante'),
    'faltas_cometidas': ('faltas_mandante', 'faltas_visitante')
}
df_scout_unified = unify_team_data(df_scout, mapping_scout)

rendimento_scout = df_scout_unified.groupby(['ano_campeonato', 'time']).agg(
    total_pontos=('pontos_ganhos', 'sum'),
    total_chutes=('chutes_totais', 'sum'),
    total_gols=('gols_pro', 'sum')
).reset_index()

rendimento_scout['eficiencia'] = (rendimento_scout['total_gols'] / rendimento_scout['total_chutes']) * 100

sns.regplot(data=rendimento_scout, x='total_chutes', y='total_pontos')
plt.title('Relação entre Total de Chutes e Pontos (2020-2023)')
plt.show()

### B. Permanência de Treinadores vs. Rendimento

In [ ]:
mapping_gestao = {
    'tecnico': ('tecnico_mandante', 'tecnico_visitante'),
    'idade_media': ('idade_media_titular_mandante', 'idade_media_titular_visitante')
}
df_gestao_unified = unify_team_data(df_gestao, mapping_gestao)

# Identificando Eras de Treinadores
df_gestao_unified = df_gestao_unified.sort_values(['ano_campeonato', 'time', 'data'])
df_gestao_unified['troca'] = (df_gestao_unified['tecnico'] != df_gestao_unified['tecnico'].shift(1)) | \
                             (df_gestao_unified['time'] != df_gestao_unified['time'].shift(1))
df_gestao_unified['era_id'] = df_gestao_unified['troca'].cumsum()

trabalhos = df_gestao_unified.groupby(['era_id', 'tecnico', 'time', 'ano_campeonato']).agg(
    jogos=('pontos_ganhos', 'count'),
    pontos=('pontos_ganhos', 'sum')
).reset_index()
trabalhos['PPM'] = trabalhos['pontos'] / trabalhos['jogos']

def categorize_duration(n):
    if n < 10: return 'Curta (<10)'
    if n <= 25: return 'Média (10-25)'
    return 'Longa (>25)'

trabalhos['permanencia'] = trabalhos['jogos'].apply(categorize_duration)

sns.boxplot(data=trabalhos, x='permanencia', y='PPM', order=['Curta (<10)', 'Média (10-25)', 'Longa (>25)'])
plt.title('Aproveitamento (PPM) vs. Tempo de Permanência')
plt.show()

### C. Faltas Cometidas vs. Rendimento

In [ ]:
mapping_scout_extended = {
    'chutes_totais': ('chutes_mandante', 'chutes_visitante'),
    'faltas_cometidas': ('faltas_mandante', 'faltas_visitante'),
    'defesas': ('defesas_mandante', 'defesas_visitante')
}
df_scout_unified = unify_team_data(df_scout, mapping_scout_extended)

rendimento_faltas = df_scout_unified.groupby(['ano_campeonato', 'time']).agg(
    total_pontos=('pontos_ganhos', 'sum'),
    total_faltas=('faltas_cometidas', 'sum')
).reset_index()

# Classificação por faixas
rendimento_faltas['rank'] = rendimento_faltas.groupby('ano_campeonato')['total_pontos'].rank(method='first', ascending=False)
rendimento_faltas['faixa'] = np.where(rendimento_faltas['rank'] <= 4, 'G4', 
                                      np.where(rendimento_faltas['rank'] >= 17, 'Z4', 'Meio'))

sns.barplot(data=rendimento_faltas, x='faixa', y='total_faltas', order=['G4', 'Meio', 'Z4'])
plt.title('Média de Faltas por Faixa de Classificação (2020-2023)')
plt.show()

### D. Idade Média vs. Rendimento

In [ ]:
rendimento_idade = df_gestao_unified.groupby(['ano_campeonato', 'time']).agg(
    total_pontos=('pontos_ganhos', 'sum'),
    idade_media=('idade_media', 'mean')
).reset_index()

rendimento_idade['rank'] = rendimento_idade.groupby('ano_campeonato')['total_pontos'].rank(method='first', ascending=False)
rendimento_idade['grupo'] = np.where(rendimento_idade['rank'] <= 4, 'G4', 'Restante')

sns.lineplot(data=rendimento_idade, x='ano_campeonato', y='idade_media', hue='grupo', marker='o')
plt.title('Evolução da Idade Média: G4 vs Restante (2007-2023)')
plt.xticks(rendimento_idade['ano_campeonato'].unique(), rotation=45)
plt.show()

## 5. Conclusões

1. **Volume vs. Eficiência:** A quantidade bruta de chutes não é o principal fator de sucesso, mas sim a eficiência na conversão.
2. **Estabilidade Técnica:** Trabalhos de longa duração (>25 jogos) apresentam PPM significativamente superior a trocas constantes.
3. **Disciplina:** Times que cometem mais faltas tendem a estar nas posições inferiores da tabela (Z4), indicando que faltas excessivas são reflexo de deficiência técnica.
4. **Idade:** Não há uma diferença significativa na idade média entre campeões e rebaixados, mantendo-se o padrão em torno de 27 anos.